<a href="https://colab.research.google.com/github/ArtyomShabunin/SMOPA-25/blob/main/lesson_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://prana-system.com/files/110/rds_color_full.png" alt="tot image" width="300"  align="center"/> &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;
<img src="https://mpei.ru/AboutUniverse/OficialInfo/Attributes/PublishingImages/logo1.jpg" alt="mpei image" width="200" align="center"/>
<img src="https://mpei.ru/Structure/Universe/tanpe/structure/tfhe/PublishingImages/tot.png" alt="tot image" width="100"  align="center"/>

---

# **??????? ????????? ???????? ? ???????????? ????????? ? ???????? ? ?????????????? ??????????**  

# ***???????????? ???????***


---

# ??????? ?10
# XGBoost
**23 ?????? 2025?.**

XGBoost (Extreme Gradient Boosting) ? ??? ?????? ? ??????????? ?????????? ????????? ????????, ?????????? ?? ?????? **???????????? ????????**. ??? ???????????? ??? ??????? ????? ?????????????, ????????? ? ????????????.  

**??????????? ???????** ? ??? ??????? **????????? ????????** ??? ?????????? ??????????????? ???????. ??? ????????? ? ?????? **??????????? ???????**, ??? ????????? ???????? ??????? (?????? ???????? ???????) ???????????? ? ???? ????????? ??????.

???????? ???? ? ??????? ?????? **???????????????**, ?????? ?? ??????? ???????? **????????? ??????** ??????????, ????????? **??????????? ?????** ??? ??????????? ??????? ??????.

### ???????? ??????????? XGBoost:
- **??????????? ???????** ? ??? ????? ???????????????, ??? ??????? ????????? ?????? ??????? (?????? ???????? ???????) ???????????? ??? ???????????? ??????? ??????.
- **?????????????** ? XGBoost ????????????? ?? ???????? ? ?????????? ??????????? ???????, ????? ??? ???????????? ?????????? ? ??????????? ????????.
- **????????? ?????????????** ? ?????????? L1 ? L2 ????????????? ???????? ????????????? ????????????.
- **????????? ??????????? ????????** ? ?????????? ????? ???? "?????????", ???? ????????? ??????????? ???????? ??? ???????? ????????.
- **????????** ? ???????????? ????????? ???? ?????, ????????? ???????????????, ? ????? ???????? ??? ? ??????????, ??? ? ? ????? ???????? ???????? ??????.

### ??? ????????????:
- ???????????? ?? ????????? ???????? (????????, Kaggle)
- ?????????? ??????
- ??????????? ? ????????
- ???????????? ???????????? ????????????
- ????????????? ?????????

C????? ?????????? ??????????? XGBoost ? ???????????? ???????? ?? ???????? ????????

1. **LightGBM (Microsoft)**
   - ??????? XGBoost ?? ??????? ??????.
   - ?????????? histogram-based ???????? ? leaf-wise ???? ??????.
   - ????? ??? ????? ? ??????? ??????????? ????????? ? ??????????????? ???????.

2. **CatBoost (Yandex)**
   - ?????? ???????? ? ??????????????? ?????????? ??? ????????????? one-hot encoding.
   - ??????? ????????????.
   - ????? ? ?????????????: ?????? ?????????? ??? ?????????.

3. **HistGradientBoosting (sklearn)**
   - ?????????? ?????????? ???????? ? ????????????? ? Scikit-learn (`sklearn.ensemble.HistGradientBoostingClassifier/Regressor`).
   - ??????? ? ???????????? natively missing values.
   - ??????? ???????????? ??? ????????? ????????? ?????????.


In [ ]:
import numpy as np
import pandas as pd

from sklearn import preprocessing
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, ConfusionMatrixDisplay
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, r2_score
import matplotlib.pyplot as plt

import ipywidgets as widgets
from IPython.display import display, clear_output

from tqdm import tqdm

## ?????????????? ????????????? ? XGBoost
### ???????? ? ????????????? ??????

In [ ]:
# import gdown
# import warnings
# warnings.filterwarnings('ignore')
# gdown.download('https://drive.google.com/uc?id=1j54o4pHTm3HvaYTEtv_i4hOJGy5yNeZZ', verify=False)

data = pd.read_parquet("./data_modes.gzip")

In [ ]:
data.head()

????????

In [ ]:
use_columns = ['GTA1.DBinPU.Alzzo', 'GTA1.DBinPU.Bo', 'GTA1.DBinPU.DlPkf',
               'GTA1.DBinPU.DlPtgft', 'GTA1.DBinPU.DlPvf', 'GTA1.DBinPU.fi',
               'GTA1.DBinPU.hmGTD', 'GTA1.DBinPU.hmTG', 'GTA1.DBinPU.P1mvhTG',
               'GTA1.DBinPU.Pk', 'GTA1.DBinPU.Pmvh', 'GTA1.DBinPU.PmvhMOGTD',
               'GTA1.DBinPU.PmvhMOTG', 'GTA1.DBinPU.PmvyhMOGTD',
               'GTA1.DBinPU.PmvyhMOTG', 'GTA1.DBinPU.Prazrjag_navhode',
               'GTA1.DBinPU.Ptgpd', 'GTA1.DBinPU.Ptgvh', 'GTA1.DBinPU.Pvh',
               'GTA1.DBinPU.Pvyhlg', 'GTA1.DBinPU.Qtg', 'GTA1.DBinPU.Tk',
               'GTA1.DBinPU.Tn', 'GTA1.DBinPU.Tt', 'GTA1.DBinPU.Tvh1',
               'GTA1.DBinPU.Pzad']

X = data.loc[:,use_columns]

??????? ??????????

In [ ]:
data['target'] = data[[
    'full_power_mode', 'partial_power_mode',
    'increas_power_mode', 'decreas_power_mode', 'start_up_mode',
    'shutdown_mode', 'stopped_state_mode']].idxmax(axis=1)
y = data.loc[:, ['target']]

In [ ]:
y.value_counts()

?????????? ?? ???????? ? ????????????? ???????

In [ ]:
from sklearn.model_selection import train_test_split
# ????????? ? ?????? ?????????? ???????
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

In [ ]:
y_train.value_counts()

In [ ]:
y_test.value_counts()

???????????? ??????

In [ ]:
!pip install imblearn

Undersampling

In [ ]:
from imblearn.under_sampling import RandomUnderSampler
sampling_strategy = {
    "full_power_mode": 1000,
    "stopped_state_mode": 1000,
    "partial_power_mode": 1000
}
rus = RandomUnderSampler(sampling_strategy=sampling_strategy, random_state=42)
X_train_resampled, y_train_resampled = rus.fit_resample(X_train, y_train)

In [ ]:
y_train_resampled.value_counts()

In [ ]:
sampling_strategy = {
    "full_power_mode": 100,
    "stopped_state_mode": 100,
    "partial_power_mode": 100
}
rus = RandomUnderSampler(sampling_strategy=sampling_strategy, random_state=42)
X_test_resampled, y_test_resampled = rus.fit_resample(X_test, y_test)

In [ ]:
y_test_resampled.value_counts()

Oversampling

In [ ]:
from imblearn.over_sampling import SMOTE
X_train_resampled, y_train_resampled = SMOTE().fit_resample(X_train_resampled, y_train_resampled)

In [ ]:
y_train_resampled.value_counts()

XGBoost ? ??? ???????? ?? ???? ???????? ???????. ? ??????? ??????? ? ??????? ?? ?? ?????? **?? ????????????? ? ???????? ?????????** (? ??????? ?? ??????? ???? ???????? ?????????, SVM ??? ????????? ?????).

In [ ]:
y_train_resampled

LabelEncoding

In [ ]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()

In [ ]:
y_train_resampled_encoded = encoder.fit_transform(y_train_resampled.values[:,0])
y_test_resampled_encoded = encoder.transform(y_test_resampled.values[:,0])

In [ ]:
y_test_resampled_encoded

In [ ]:
encoder.classes_

### ????????????? ? ???????? ??????

In [ ]:
!pip install xgboost
import xgboost as xgb

`xgb.XGBClassifier` ? ??? ????? ?? ?????????? **XGBoost**, ??????????????? ??? ??????? **????? ?????????????** ? ??????? ????????? **???????????? ???????? ?? ???????? ???????**.

- ?????? ???????? ???????? ??????? (?? ????????? ? ??????? ?????????????),
- ?? ?????? ???? ????????? ??????, ??????? ?????????? ?????? ??????????,
- ???????????? ????????? **??????? ?????? (objective)**.

---

**??????????? ???:**

- **???????? ?????????????** (????????, "??/???", "????/?? ????")
- **?????????????? ?????????????** (????????, ????????????? ????, ????? ???????? ? ?.?.)

---

???????? **`objective`** ? XGBoost `XGBClassifier` ?????????? **??????? ??????**, ??????? ???????????? ??????, ? ???????? ??????? ?? ???? ?????? ? ?????????????, ?????????, ???????????? ? ?.?.


---

**??? ????? ?????????????? ?????????????:**

- `'multi:softmax'` ? ?????????????? ?????????????, ?????????? **????? ??????** ????????.  
  ????????? ???????? `num_class`.
- `'multi:softprob'` ? ?????????????? ?????????????, ?????????? **?????? ???????????? ?? ??????? ??????**.  
  ????? ??????? `num_class`.

---

**???????? ????????? ???????????????:**

- **`num_class=7`**  
  ?????????? ?????????? ??????? ? ?????? ?????????????. ? ?????? ?????? ? 7 ???????.

- **`max_depth=6`**  
  ???????????? ??????? ??????? ??????.  
  ???????? ??????? ????? ???????????? ??????? ???????????, ?? ????? ???????? ???? ????????????.

- **`learning_rate=0.1`**
  ???????? ????????, ???????? ????? ??????? ?????? ? ???????? ???????.  
  ??????? ???????? ??????? ?????? ???????? (`n_estimators`) ??? ??????? ??????????.

- **`n_estimators=100`**  
  ?????????? ???????? (???????? ????????).  
  ???????? ?????? ????? ???????????? ???? 100 ?????? ???????.

- **`subsample=0.8`**  
  ???? ???????? ????????? ???????? ??? ???????? ??????? ??????.  
  ???????? <1.0 ????? ?????? ? ?????? ? ?????????????.

- **`colsample_bytree=0.8`**  
  ???? ???????? ????????? ?????????, ???????????? ??? ?????????? ??????? ??????.  
  ????? ??????? ???? ????????????, ???????? ??? ??????? ????? ?????????.

- **`random_state=42`**  
  ????????? ????????? ????????? ?????, ??????????? ????????????????? ???????????.


In [ ]:
from sklearn.utils import shuffle

classifier = xgb.XGBClassifier(
    objective='multi:softmax',
    # objective='multi:softprob',
    num_class=7,
    max_depth=6,
    learning_rate=0.1,
    n_estimators=100,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

X_shuffled, y_shuffled = shuffle(
    X_train_resampled, y_train_resampled_encoded, random_state=42)

classifier.fit(X_shuffled, y_shuffled)

In [ ]:
y_pred = classifier.predict(X_test_resampled)

In [ ]:
y_pred

In [ ]:
classifier.predict_proba(X_test_resampled)

In [ ]:
np.argmax(classifier.predict_proba(X_test_resampled), axis=1)
# np.max(classifier.predict_proba(X_test_resampled), axis=1)

### ?????? ???????? ??????
??????? ???????????

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

conf_mat = confusion_matrix(y_test_resampled_encoded, y_pred)
ConfusionMatrixDisplay(conf_mat, display_labels=encoder.classes_).plot()
plt.xticks(rotation=90)
plt.show()

In [ ]:
print(classification_report(y_test_resampled_encoded, y_pred, target_names=encoder.classes_))

### ?????? ???????? ???????????????
**RandomizedSearchCV**

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

# ???????? ?????? XGB
classifier =  xgb.XGBClassifier(
    objective='multi:softmax', num_class=7, random_state=42)

# ????? ???????????????
param_grid = {
    'max_depth': [4, 6, 8],
    'learning_rate': [0.01, 0.1, 0.2],
    'n_estimators': [100, 200],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0]
}

# ????????????? RandomizedSearchCV ??? ??????? ??????????
random_search = RandomizedSearchCV(
    estimator=classifier, param_distributions=param_grid, n_iter=10, cv=5,
    n_jobs=-1, verbose=2, random_state=42)

# ???????? ?????? ? RandomizedSearchCV
random_search.fit(X_shuffled, y_shuffled)

# ????? ?????? ??????????
print("?????? ?????????:", random_search.best_params_)

In [ ]:
# ?????? ?????? ?? ???????? ??????
best_classifier = random_search.best_estimator_
# print("???????? ?? ???????? ???????:", best_classifier.score(X_test_resampled, y_test_resampled_encoded))
y_pred = best_classifier.predict(X_test_resampled)

### ?????? ???????? ??????

In [ ]:
conf_mat = confusion_matrix(y_test_resampled_encoded, y_pred)
ConfusionMatrixDisplay(conf_mat, display_labels=encoder.classes_).plot()
plt.xticks(rotation=90)
plt.show()

In [ ]:
print(classification_report(y_test_resampled_encoded, y_pred, target_names=encoder.classes_))

## ??????????????? ? XGBoost

In [ ]:
# import gdown
# import warnings
# warnings.filterwarnings('ignore')
# url = "https://drive.google.com/drive/folders/1RtrAevJUYSgTbp0YUztxEBB8_VcvjgGH?usp=drive_link"
# gdown.download_folder(url, quiet=True, verify=False)

In [ ]:
import glob
parquetFileList = glob.glob(f'./option_0/*.gzip')

In [ ]:
df_list = []

for file in tqdm(parquetFileList):
    df = pd.read_parquet(file)
    df_list.append(df)

data = pd.concat(df_list, axis=0).sort_index().ffill().drop_duplicates()
data = data.dropna()

In [ ]:
import seaborn as sns
sns.set_theme(style="whitegrid")

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(10, 6), constrained_layout=True)

ax1.plot(data['GTA1.DBinPU.P'].index, data['GTA1.DBinPU.P'].values,
         linestyle='none', marker='.', markersize=1);
ax1.set_ylabel("????????, ???");
ax1.set_ylim([5700,6100]);
ax1.tick_params(labelbottom=False)

ax2.plot(data['GTA1.DBinPU.Qtg'].index, data['GTA1.DBinPU.Qtg'].values,
         linestyle='none', marker='.', markersize=1);
ax2.set_ylabel("?????? ???????, ?3/?");
ax2.set_ylim([1500,2500]);
ax2.tick_params(labelbottom=False)

ax3.plot(data['GTA1.DBinPU.Tt'].index, data['GTA1.DBinPU.Tt'].values,
         linestyle='none', marker='.', markersize=1);
ax3.set_ylabel("???????????\n??. ?????, $^\\circ$?");
# ax3.set_ylim([300,600]);
ax3.tick_params(rotation=45);

### ????????????? ??????

???????? ? ??????????? ??????

In [ ]:
df = pd.DataFrame(data['GTA1.DBinPU.Tt'])
df = df.resample('1h').mean()
df.columns = ['Tt']
df = df[df['Tt'] > 300]
df = df.resample('1h').asfreq()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df[df['Tt'].isna()]

In [ ]:
df.index.to_series().diff().max()

In [ ]:
df.loc['2023-03-05 01:00:00':'2023-03-05 08:00:00']

In [ ]:
df.plot(style='.', figsize=(10, 2), ms=3, title="??????????? ??. ?????", legend=False)
plt.ylabel("$^\\circ$?")
plt.show()

**????** ? ??? **???????? ?????????? ???? ?? ?????????? ????????? ?????**, ??????? ???????????? ??? ???????? (????) ??? ???????????? ??????? ????????.

??????? ??? ????????? ?????

In [ ]:
def create_lag_features(df, column, lags):
    """
    ??????? ???? ???????? column ? ?????????? lags ? ?????????? ????? DataFrame.
    """
    df_lagged = df.copy()
    for lag in range(1, lags + 1):
        df_lagged[f"{column}_lag_{lag}"] = df_lagged[column].shift(lag)

    return df_lagged

In [ ]:
n_lags = 24
df = create_lag_features(df, 'Tt', n_lags)

In [ ]:
df.index.to_series().diff().max()

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.dropna(inplace=True)

??????? ????? ????????? ? ?????? ??????? ????????

In [ ]:
y = df.loc[:, ['Tt']]
X = df.loc[:, [col for col in df.columns if 'lag' in col]]

???????? ?? ????????????? ? ???????? ??????? (**????????? ??????? ????? ?????????!**)

In [ ]:
train_size = int(len(df) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

### ????????????? ? ???????? ??????

In [ ]:
pred_model = xgb.XGBRegressor()
pred_model.fit(X_train, y_train)

????????????

In [ ]:
y_pred = pred_model.predict(X_test)

baseline_pred = X_test.mean(axis=1).values

### ?????? ???????? ??????

In [ ]:
# ??????
from sklearn.metrics import mean_squared_error

rmse = np.sqrt(mean_squared_error(y_test, y_pred))

baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))

print("RMSE:", round(rmse, 3))
print("Baseline RMSE:", round(baseline_rmse, 3))

In [ ]:
# ????????????
plt.plot(y_test.values, label='??????')
plt.plot(y_pred, label='??????? xgb')
plt.plot(baseline_pred, label='??????? baseline')
plt.legend()
plt.title("??????? ?????????? ???? XGBoost")
plt.show()

### ?????????? ????????????????? ??????? (???? ????? ?? ???)
?? ???? ?????? ???????? ????????????, ? ?? ???????? ???????? ?? ??????

In [ ]:
start = X_test.index[0]
end = X_test.index[-1]
timestamps = pd.date_range(start=start, end=end, freq='1H')

In [ ]:
predictions = []
lags = X_test.iloc[0].values.tolist()  # ?????? ???? ?? ?????? ?????

for _ in timestamps:
    # ???????
    y_pred = pred_model.predict(np.array([lags]))[0]
    predictions.append(y_pred)

    # ????????? ????: ???????? ????? ? ????????? ????? ???????
    lags = lags[1:] + [y_pred]

predictions = pd.DataFrame(predictions, index=timestamps, columns=['Tt'])

baseline_predictions = [X_test.iloc[0].mean() for _ in y_test.index]

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(timestamps, predictions['Tt'], label='??????? xgb')
plt.plot(df['Tt'].loc[start:].index, df['Tt'].loc[start:], linestyle='none', marker='.', markersize=5, label='??????')
plt.plot(y_test.index, baseline_predictions, label='??????? baseline')

plt.ylabel("$^\\circ$?")
plt.title('Autoregressive ??????? XGBoost')
plt.xlabel('?????? ???????')
plt.legend()
plt.grid(True)
plt.tight_layout()

plt.xticks(rotation=45)
# plt.xlim([start, start + pd.Timedelta("4d")])

plt.show()

In [ ]:
# ????? ???????
common_index = y_test.index.intersection(predictions.index)

In [ ]:
rmse = np.sqrt(
    mean_squared_error(y_test.loc[common_index], predictions.loc[common_index]))

baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_predictions))

print("RMSE:", round(rmse, 3))
print("Baseline RMSE:", round(baseline_rmse, 3))

### ?????? ? ????????????? ????????? (????? n ?????)

**sequence-to-sequence (seq2seq)** ??????? ? XGBoost  
?????? ???????? ?? ???? `input_len` ??????? ???????? ? ????????????? ????? `output_len` ???????

XGBoost **?? ????????????** ???????? ??????????? ?????, ?? ????? ??????? ??????? `output_len` ????????? ???????, ?????? ?? ??????? ????? ????????????? ???? ????????


In [ ]:
def make_seq2seq_dataset(series, input_len, output_len):
    X, Y = [], []
    for i in range(len(series) - input_len - output_len + 1):
        x_temp = series[i:i + input_len]
        y_temp = series[i + input_len:i + input_len + output_len]

        if x_temp.isna().any(): continue
        if y_temp.isna().any(): continue

        X.append(x_temp)
        Y.append(y_temp)

    return np.array(X), np.array(Y)

In [ ]:
input_len = 20
output_len = 20
X, Y = make_seq2seq_dataset(
    df.resample('1h').asfreq()['Tt'],
    input_len=input_len, output_len=output_len)

In [ ]:
X.shape, Y.shape

In [ ]:
X[0], Y[0]

?????????? ?? ???????? ? ????????????? ???????

In [ ]:
train_size = int(len(Y) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = Y[:train_size], Y[train_size:]

`MultiOutputRegressor`

??? **???????**, ??????? ????????? ???????????? ????? ??????? **????????????? ??????** (????????, `XGBRegressor`, `RandomForestRegressor`, `LinearRegression`) ??? ??????? ????? **??????????? ?????????** (multi-output regression).

In [ ]:
from sklearn.multioutput import MultiOutputRegressor

model = MultiOutputRegressor(xgb.XGBRegressor())
model.fit(X_train, y_train)  # X_train: (n_samples, 20), y_train: (n_samples, 20)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
baseline_pred = np.array([np.ones(output_len)*x.mean() for x in X_test])

In [ ]:
sample = 0

plt.plot(range(output_len), y_test[sample], label="???????? ????????")
plt.plot(range(output_len), y_pred[sample], label="??????? xgb", linestyle="--")
plt.plot(range(output_len), baseline_pred[sample], label='??????? baseline')
plt.legend()
plt.title(f"??????? ?? {output_len} ????? ??????")
plt.show()

In [ ]:
# np.sqrt(mean_squared_error(y_test[0], y_pred[0]))

In [ ]:
# np.sqrt(mean_squared_error(y_test[0], baseline_pred[0]))

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))

print("RMSE:", round(rmse, 3))
print("Baseline RMSE:", round(baseline_rmse, 3))

### ?????????? ????????????????? ??????? ? swq2seq ???????

In [ ]:
start = df[df['Tt'] == y_test[0][0]].index[0]
end = df.index[-1]
timestamps = pd.date_range(start=start, end=end, freq='1h')

In [ ]:
current_input = X_test[0]
predictions = []

# ??????? ??? ???? ??????? ????????????
n_iters = (len(timestamps) - input_len) // output_len

In [ ]:
for _ in range(n_iters):
    # ???????????? ????????? ????????
    pred = model.predict(np.array([current_input]))[0]
    predictions.extend(pred)

    # ????????? ????: ????????? input_len ????????????? ????????
    current_input = pred.tolist()[-input_len:]

In [ ]:
predictions = pd.DataFrame(
    predictions,
    index=timestamps[:len(predictions)],
    columns=['Tt'])

In [ ]:
# ????? ???????
common_index = df.index.intersection(predictions.index)

In [ ]:
baseline_predictions = [X_test[0].mean() for _ in common_index]

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(
    predictions.index, predictions['Tt'], label="??????? xgb")
plt.plot(
    df['Tt'].loc[start:].index, df['Tt'].loc[start:],
    linestyle='none', marker='.', markersize=5,
    label="???????? ????????")
plt.plot(
    df['Tt'].loc[common_index].index, baseline_predictions,
    label='??????? baseline')

plt.ylabel("$^\\circ$?")
plt.title('Autoregressive ??????? XGBoost')
plt.xlabel('?????? ???????')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.xticks(rotation=45)

# plt.xlim([start, start + pd.Timedelta("4d")])

plt.show()

In [ ]:
rmse = np.sqrt(
    mean_squared_error(
        df.loc[common_index, ['Tt']], predictions.loc[common_index]))

baseline_rmse = np.sqrt(
    mean_squared_error(df.loc[common_index, ['Tt']], baseline_predictions))

print("RMSE:", round(rmse, 3))
print("Baseline RMSE:", round(baseline_rmse, 3))